Правка дефисов в трейн

In [ ]:
def fix_conll(lines):
    fixed_lines = []
    buffer = []

    def process_buffer(buffer):
        result = []
        prev = None

        for line in buffer:
            parts = line.split("\t")
            _, token, intent, tag = parts

            if token.startswith("-") and prev:
                prev_parts = prev.split("\t")
                prev_parts[1] += token
                prev = "\t".join(prev_parts)
            else:
                if prev:
                    result.append(prev)
                prev = line

        if prev:
            result.append(prev)

        renumbered = []
        for i, line in enumerate(result, start=1):
            parts = line.split("\t")
            parts[0] = str(i)
            renumbered.append("\t".join(parts))

        return renumbered

    for line in lines:
        line = line.rstrip()

        if line.startswith("#") or line == "":
            if buffer:
                fixed_lines.extend(process_buffer(buffer))
                buffer = []
            fixed_lines.append(line)
        else:
            buffer.append(line)

    if buffer:
        fixed_lines.extend(process_buffer(buffer))

    return fixed_lines


with open("<Ваш файл>", "r", encoding="utf-8") as f:
    lines = f.readlines()

fixed = fix_conll(lines)

with open("<Имя для сохранения файла>", "w", encoding="utf-8") as f:
    f.write("\n".join(fixed))

Адаптация, список сущностей в файле `entities.py`

In [ ]:
import random
import re
import unicodedata
from collections import defaultdict
from entities import *

INPUT_FILE = "<Ваш файл>"
OUTPUT_FILE = "<Ваш файл>"
LOG_FILE = "debug_log.txt"

log = open(LOG_FILE, "w", encoding="utf-8")

def flatten(x):
    if isinstance(x, str):
        return x.split()
    result = []
    for item in x:
        if isinstance(item, list):
            result.extend(item)
        elif isinstance(item, str):
            result.extend(item.split())
        else:
            result.append(item)
    return result

city_counter = defaultdict(int)

def get_city():
    min_count = min(city_counter[" ".join(c)] for c in cities)
    candidates = [c for c in cities if city_counter[" ".join(c)] == min_count]
    city = random.choice(candidates)
    city_counter[" ".join(city)] += 1
    return city.copy()


def normalize(text):
    text = unicodedata.normalize("NFKD", text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

def is_country(word):
    norm = normalize(word)
    return any(norm.startswith(root) for root in COUNTRY_ROOTS)

def is_fahrenheit(token):
    return "фаренг" in normalize(token)


FRONT = "әеөүи"
BACK = "аоуы"

def is_front(word):
    for ch in reversed(word.lower()):
        if ch in FRONT:
            return True
        if ch in BACK:
            return False
    return False

def apply_case(word, case):
    if not case:
        return word
    if case == "loc":
        return word + ("дә" if is_front(word) else "да")
    if case == "dat":
        return word + ("гә" if is_front(word) else "га")
    return word

def detect_case(word):
    w = word.lower()
    if w.endswith(("да","дә","та","тә")):
        return "loc"
    if w.endswith(("га","гә","ка","кә")):
        return "dat"
    return None

def extract_suffix(word):

    clean = word.replace("-", "")

    suffixes = [
        "ның","нең",
        "дан","дән","тан","тән",
        "га","гә","ка","кә",
        "да","дә","та","тә"
    ]

    for suf in suffixes:
        if clean.endswith(suf):
            base = clean[:-len(suf)]
            return base, suf

    return clean, None


def normalize_suffix(word, suffix):

    last = word[-1].lower()
    voiceless = "пктсфхчшщц"

    if suffix in ["ның", "нең"]:
        return word + ("нең" if is_front(word) else "ның")

    if suffix in ["да", "дә", "та", "тә"]:
        if last in voiceless:
            return word + ("тә" if is_front(word) else "та")
        return word + ("дә" if is_front(word) else "да")

    if suffix in ["га", "гә", "ка", "кә"]:
        if last in voiceless:
            return word + ("кә" if is_front(word) else "ка")
        return word + ("гә" if is_front(word) else "га")

    if suffix in ["дан", "дән", "тан", "тән"]:
        if last in voiceless:
            return word + ("тән" if is_front(word) else "тан")
        return word + ("дән" if is_front(word) else "дан")

    return word + suffix


def build_tags(slot, new_len):
    if new_len == 1:
        return ["B-" + slot]
    return ["B-" + slot] + ["I-" + slot] * (new_len - 1)


SAFE_SLOTS = {
    "location", "playlist", "restaurant_name", "movie_name",
    "object_name", "artist", "entity_name", "song", "album",
    "service", "temperatureUnit"
}

def process_entity(tokens, tags, i, intent):

    slot = tags[i].split("-", 1)[1]
    base_slot = slot.split("/")[-1]

    j = i
    entity_tokens = []
    entity_tags = []

    while j < len(tokens) and (tags[j].startswith("I-") or j == i):
        entity_tokens.append(tokens[j])
        entity_tags.append(tags[j])
        j += 1

    log.write(f"\nENTITY: {' '.join(entity_tokens)} | SLOT: {slot}\n")

    if base_slot not in SAFE_SLOTS:
        return entity_tokens, entity_tags, j

    if base_slot == "temperatureUnit":

        if any(is_fahrenheit(t) for t in entity_tokens):

            last = entity_tokens[-1]
            base, suffix = extract_suffix(last)

            word = "цельсий"

            if suffix:
                word = normalize_suffix(word, suffix)
            else:
                case = detect_case(last)
                if case:
                    word = apply_case(word, case)

            return [word], build_tags(slot, 1), j

    if base_slot == "location":

        last_token = entity_tokens[-1]
        base, suffix = extract_suffix(last_token)

        if any(is_country(t) for t in entity_tokens):

            new_tokens = []

            if len(entity_tokens) > 1:
                new_city = get_city()
                new_tokens.extend(new_city)

            country = "Россия"

            if suffix:
                country = normalize_suffix(country, suffix)
            else:
                case = detect_case(base)
                if case:
                    country = apply_case(country, case)

            new_tokens.append(country)
            return new_tokens, build_tags(slot, len(new_tokens)), j

        new_city = get_city()
        new_tokens = new_city.copy()

        if suffix:
            new_tokens[-1] = normalize_suffix(new_tokens[-1], suffix)
        else:
            case = detect_case(base)
            if case:
                new_tokens[-1] = apply_case(new_tokens[-1], case)

        return new_tokens, build_tags(slot, len(new_tokens)), j

    return entity_tokens, entity_tags, j

def process_sentence(tokens, tags, intent):

    new_tokens = []
    new_tags = []

    i = 0

    while i < len(tokens):

        if tags[i].startswith("B-"):
            nt, tg, ni = process_entity(tokens, tags, i, intent)
            new_tokens.extend(nt)
            new_tags.extend(tg)
            i = ni
            continue

        token = tokens[i]

        if is_fahrenheit(token):
            token = "цельсий"

        new_tokens.append(token)
        new_tags.append(tags[i])

        i += 1

    return new_tokens, new_tags

def flush_sentence(tokens, tags, intent):

    if not tokens:
        return []

    new_tokens, new_tags = process_sentence(tokens, tags, intent)

    result = []
    result.append(f"# text: {' '.join(new_tokens)}")

    for idx, (t, tg) in enumerate(zip(new_tokens, new_tags), start=1):
        result.append(f"{idx}\t{t}\t{intent}\t{tg}")

    result.append("")
    return result

def process_file(lines):

    output = []
    tokens = []
    tags = []
    intent = None

    for line in lines:

        if line.startswith("# text"):
            continue

        if line.startswith("# id:"):
            output.extend(flush_sentence(tokens, tags, intent))
            tokens, tags, intent = [], [], None
            output.append(line)
            continue

        if line.startswith("# intent:"):
            intent = line.split(":")[1].strip()
            output.append(line)
            continue

        if line.startswith("#"):
            output.append(line)
            continue

        if not line.strip():
            output.extend(flush_sentence(tokens, tags, intent))
            tokens, tags, intent = [], [], None
            continue

        parts = line.split("\t")
        if len(parts) < 4:
            continue

        tokens.append(parts[1])
        tags.append(parts[3])

    output.extend(flush_sentence(tokens, tags, intent))
    return output


with open(INPUT_FILE, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")

processed = process_file(lines)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(processed))

log.close()

print("CITY COUNTER:", dict(city_counter))

В правильный порядок ставит intent, slot, text

In [ ]:
def fix_order(input_file, output_file):

    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()

    paragraphs = content.strip().split("\n\n")
    fixed_paragraphs = []

    for p_idx, paragraph in enumerate(paragraphs):

        lines = paragraph.split("\n")

        id_line = None
        text_line = None
        intent_line = None
        slots_line = None
        other_lines = []

        for line in lines:
            if line.startswith("# id:"):
                id_line = line
            elif line.startswith("# text:"):
                text_line = line
            elif line.startswith("# intent:"):
                intent_line = line
            elif line.startswith("# slots:"):
                slots_line = line
            else:
                other_lines.append(line)

        new_block = []

        if id_line:
            new_block.append(id_line)

        if text_line:
            new_block.append(text_line)

        if intent_line:
            new_block.append(intent_line)

        if slots_line:
            new_block.append(slots_line)

        new_block.extend(other_lines)

        fixed_paragraphs.append("\n".join(new_block))

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(fixed_paragraphs))


fix_order("<Ваш файл>",
          "<Имя файл для сохранения результата>")